# Descriptive Statistics

Summarize, visualize, and understand data before any modeling — the foundation of all data work.

**Topics:** Central tendency, spread, shape, outlier detection, correlation, covariance, visual EDA

## 1. Central Tendency — Mean, Median, Mode

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# Dataset: house prices (right-skewed, realistic)
prices = np.concatenate([
    np.random.normal(350_000, 80_000, 900),  # typical homes
    np.random.normal(1_200_000, 200_000, 100),  # luxury homes (outliers)
]).clip(min=100_000)

mean_price = np.mean(prices)
median_price = np.median(prices)
mode_result = stats.mode(prices.round(-4))

print('House Price Distribution:')
print(f'  Mean:   ${mean_price:>10,.0f}  ← pulled up by luxury homes')
print(f'  Median: ${median_price:>10,.0f}  ← robust to luxury outliers')
print(f'  n = {len(prices):,}')
print()

# Trimmed mean — robust alternative to mean
trimmed_mean = stats.trim_mean(prices, proportiontocut=0.05)  # trim 5% each end
print(f'  Trimmed mean (5%): ${trimmed_mean:>10,.0f}')
print()

# When to use each:
guidance = [
    ('Mean', 'Symmetric, no outliers', 'Income with extreme wealth, skewed data'),
    ('Median', 'Skewed data, outliers present', 'Symmetric distributions'),
    ('Mode', 'Categorical or discrete data', 'Continuous data'),
    ('Trimmed mean', 'Contaminated data', 'Need all data points counted'),
    ('Geometric mean', 'Ratios, growth rates, log-normal', 'Negative values present'),
]
print('Choosing the right central tendency measure:')
print(f'{"Measure":<15} {"Use when":<35} {"Avoid when"}')
print('-' * 75)
for measure, use, avoid in guidance:
    print(f'{measure:<15} {use:<35} {avoid}')

## 2. Spread — Variance, Standard Deviation, IQR

In [ ]:
# Two datasets with same mean, different spread
a = np.random.normal(100, 5, 1000)    # tight distribution
b = np.random.normal(100, 25, 1000)   # wide distribution

for name, data in [('A (tight)', a), ('B (wide)', b)]:
    q1, q3 = np.percentile(data, [25, 75])
    print(f'Dataset {name}:')
    print(f'  Mean:   {np.mean(data):.1f}')
    print(f'  Std:    {np.std(data):.1f}    ← same units as data')
    print(f'  Var:    {np.var(data):.1f}    ← squared units (less intuitive)')
    print(f'  IQR:    {q3-q1:.1f}    ← robust: Q3 - Q1')
    print(f'  Range:  {data.max()-data.min():.1f}  ← non-robust (affected by single outlier)')
    print()

# Coefficient of variation: compare spread across different-scaled variables
cv = lambda x: np.std(x) / np.mean(x) * 100
salary = np.random.normal(80_000, 15_000, 500)
age = np.random.normal(35, 8, 500)

print('Coefficient of Variation (CV = std/mean × 100):')
print(f'  Salary: CV = {cv(salary):.1f}%')
print(f'  Age:    CV = {cv(age):.1f}%')
print('  Higher CV = more relative variability')
print()

# Standard error vs standard deviation
n_samples = [10, 100, 1000, 10000]
print('Standard Error = std / sqrt(n) — uncertainty of the mean:')
pop_std = 15_000
for n in n_samples:
    se = pop_std / np.sqrt(n)
    print(f'  n={n:>6}: SE = ${se:,.0f}  ← 95% CI width: ±${1.96*se:,.0f}')

## 3. Shape — Skewness and Kurtosis

In [ ]:
import scipy.stats as stats

# Generate distributions with different shapes
distributions = {
    'Normal':       np.random.normal(0, 1, 10000),
    'Right skewed': stats.lognormal.rvs(0, 1, 10000),  # income-like
    'Left skewed':  -stats.lognormal.rvs(0, 1, 10000), # failure time
    'Heavy tails':  stats.t.rvs(3, size=10000),         # t-distribution, df=3
    'Bimodal':      np.concatenate([np.random.normal(-2, 0.5, 5000), np.random.normal(2, 0.5, 5000)]),
}

print(f'{"Distribution":<15} {"Mean":>8} {"Median":>8} {"Skewness":>10} {"Kurtosis":>10} {"Interpretation"}')
print('-' * 80)
for name, data in distributions.items():
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)  # excess kurtosis (normal = 0)
    interp = ''
    if abs(skew) > 1: interp += 'strongly skewed'
    elif abs(skew) > 0.5: interp += 'moderately skewed'
    else: interp += 'approx symmetric'
    if kurt > 2: interp += ', heavy tails'
    print(f'{name:<15} {np.mean(data):>8.2f} {np.median(data):>8.2f} {skew:>10.2f} {kurt:>10.2f}  {interp}')
print()
print('Skewness: >0 right tail, <0 left tail, ~0 symmetric')
print('Kurtosis: >0 heavy tails (leptokurtic), <0 light tails (platykurtic)')
print()
print('ML implication: skewed features → log-transform before regression')
print('Heavy tails → use robust models (tree-based) vs linear models')

## 4. Outlier Detection Methods

In [ ]:
from scipy import stats

np.random.seed(42)
data = np.concatenate([np.random.normal(50, 10, 200), [150, 160, -30, 200]])  # 4 outliers

def iqr_outliers(x: np.ndarray, k: float = 1.5) -> np.ndarray:
    """IQR method — robust, works for skewed distributions."""
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return (x < q1 - k * iqr) | (x > q3 + k * iqr)

def zscore_outliers(x: np.ndarray, threshold: float = 3.0) -> np.ndarray:
    """Z-score method — assumes normal distribution."""
    return np.abs(stats.zscore(x)) > threshold

def modified_zscore_outliers(x: np.ndarray, threshold: float = 3.5) -> np.ndarray:
    """Modified Z-score using median — more robust than standard z-score."""
    median = np.median(x)
    mad = np.median(np.abs(x - median))  # Median Absolute Deviation
    modified_z = 0.6745 * (x - median) / mad
    return np.abs(modified_z) > threshold

methods = [
    ('IQR (k=1.5)', iqr_outliers(data)),
    ('Z-score (>3σ)', zscore_outliers(data)),
    ('Modified Z-score', modified_zscore_outliers(data)),
]

print(f'Data: {len(data)} points, 4 injected outliers: [150, 160, -30, 200]')
print()
print(f'{"Method":<22} {"Outliers found":<18} {"Values"}')
print('-' * 65)
for name, mask in methods:
    detected = data[mask]
    print(f'{name:<22} {mask.sum():<18} {sorted(detected)}')

print()
print('When to remove vs keep outliers:')
print('  Remove: data entry errors, sensor malfunctions')
print('  Keep:   real phenomena (fraud transactions, viral posts)')
print('  Document: always explain what you did and why')

## 5. Correlation, Covariance, and Relationships

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 500

# Generate correlated features
age = np.random.normal(35, 10, n).clip(18, 70)
income = 20_000 + age * 1_500 + np.random.normal(0, 15_000, n)  # +corr with age
satisfaction = 8 - income / 100_000 + np.random.normal(0, 1, n)   # -corr with income
random_col = np.random.randn(n)  # no correlation

df = pd.DataFrame({'age': age, 'income': income, 'satisfaction': satisfaction, 'random': random_col})

# Pearson: linear correlation
print('Pearson Correlation Matrix:')
print(df.corr('pearson').round(3))
print()

# Spearman: monotonic (non-linear) correlation — rank-based
pearson_r, _ = stats.pearsonr(age, income)
spearman_r, _ = stats.spearmanr(age, income)
print(f'Age-Income: Pearson={pearson_r:.3f}, Spearman={spearman_r:.3f}')
print('(Similar → relationship is linear)')
print()

# Point-biserial: continuous vs binary
churned = (satisfaction < 6).astype(int)
r_pb, p_val = stats.pointbiserialr(churned, income)
print(f'Churn-Income: r_pb={r_pb:.3f}, p={p_val:.4f}')
print()

print('Correlation strength guide:')
for r, strength in [('|r| < 0.2', 'Negligible'), ('0.2-0.4', 'Weak'), ('0.4-0.6', 'Moderate'), ('0.6-0.8', 'Strong'), ('|r| > 0.8', 'Very strong')]:
    print(f'  {r:<12}: {strength}')
print()
print('WARNING: Correlation ≠ Causation. Spurious correlations are everywhere.')
print('Example: Ice cream sales correlate with drowning (both caused by summer).')

## Exercises

1. **Anscombe's Quartet:** Load the classic Anscombe's Quartet dataset (4 datasets with identical summary statistics but very different distributions). Compute mean, std, and correlation for each. Then visualize all four. What does this teach you about always plotting data?

2. **Robust statistics pipeline:** Given a dataset with ~10% random outlier contamination, compare the performance of mean/std vs median/IQR for estimating the true distribution parameters. Use bootstrapping to quantify estimation uncertainty.

3. **Correlation detective:** Generate 3 pairs of variables with (a) linear correlation, (b) non-linear (quadratic) relationship with low Pearson r, (c) no relationship. Show that Pearson fails on case (b) but Spearman succeeds. Implement mutual information as an alternative.